### TP-1

0. Add the 'ipykernel' kernel
  ```
  python 
  uv add ipykernel
  ```
1. Anonymize the information:
   - Replace the values in the "operator_name" column with "ANONYMOUS"
   - Replace the values in the "operator_badge" column with "ANONYMOUS" 
2. Set an output directory with a date in the format "YYYYMMDDHHMM"
   - Add libraries: 
     - datetime
     - os
   - Create the output directory in the correct format. 
3. Possible evolution: Add random information using the uuid library

In [21]:
import pandas as pd
from datetime import datetime
import os


# 1. Load the CSV file
file_path = "./artifacts/ingestions/datas/releves_incidents.csv"
df = pd.read_csv(file_path)

# 2. Anonymize the 'operator_name' and 'operator_badge' columns
# Replace with a fixed value (e.g., "ANONYMISED")
df["operator_name"] = "ANONYMOUS"
df["operator_badge"] = "ANONYMOUS"

# 3. Create a directory with the date in YYYYMMDD format
date_time_output = datetime.now().strftime("%Y%m%d%H%M")  # Format YYYYMMDDHHMM
output_dir = f"./artifacts/ingestions/incidents/{date_time_output}"

# Create the directory if it does not exist
os.makedirs(output_dir, exist_ok=True)

# 4. Save the anonymized file in the directory
output_file_path = os.path.join(output_dir, "releves_incidents_anonymised.csv")
df.to_csv(output_file_path, index=False)

print(f"Anonymized file saved at: {output_file_path}")

Anonymized file saved at: ./artifacts/ingestions/incidents/202606161613\releves_incidents_anonymised.csv


4. Add library "matplotlib"
```
python
uv add matplotlib
```
5. Ask to générate graph by day.
   - Add as input environment variable file location [INPUT_DATA_DIR] 

In [27]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ----------------------------
# 1. Configure paths and environment variables
# ----------------------------
# Environment variable for the input directory
input_dir_env_var = "INPUT_DATA_DIR"
input_dir = os.getenv(input_dir_env_var, default=".")  # Default to current directory if not set

# Name of the file to search for
file_name = "releves_incidents_anonymised.csv"
file_path = None

# Search for the file in the input directory
for root, dirs, files in os.walk(input_dir):
    if file_name in files:
        file_path = os.path.join(root, file_name)
        break

if not file_path:
    raise FileNotFoundError(f"File '{file_name}' not found in directory: {input_dir}")

print(f"Found file: {file_path}")

# ----------------------------
# 2. Load the CSV file
# ----------------------------
df = pd.read_csv(file_path)

# ----------------------------
# 3. Configure column names (adjust according to your CSV)
# ----------------------------
date_column = "date"          # Column containing date/time (e.g., "incident_datetime")
shift_column = "shift"        # Column containing the shift (e.g., "work_shift", "shift_type")

# Check if the columns exist
if date_column not in df.columns:
    raise ValueError(f"Column '{date_column}' not found in the CSV file. Available columns: {df.columns.tolist()}")
if shift_column not in df.columns:
    raise ValueError(f"Column '{shift_column}' not found in the CSV file. Available columns: {df.columns.tolist()}")

# Convert the date column to datetime
df[date_column] = pd.to_datetime(df[date_column], errors="coerce")

# ----------------------------
# 4. Create output directory for graphs
# ----------------------------
output_graph_dir = os.path.join(input_dir, "graphs")
os.makedirs(output_graph_dir, exist_ok=True)

# ----------------------------
# 5. Generate the graphs
# ----------------------------

# --- Graph 1: Distribution by Day ---
df["day"] = df[date_column].dt.date  # Extract date (without time)
daily_distribution = df["day"].value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(daily_distribution.index, daily_distribution.values, marker="o", linestyle="-", color="b")
plt.title("Distribution of Incidents by Day", fontsize=14)
plt.xlabel("Day", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_graph_dir, "incidents_by_day.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 2: Distribution by Week ---
df["week"] = df[date_column].dt.to_period("W").astype(str)  # Extract week (format: YYYY-WXX)
weekly_distribution = df["week"].value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(weekly_distribution.index.astype(str), weekly_distribution.values, marker="o", linestyle="-", color="g")
plt.title("Distribution of Incidents by Week", fontsize=14)
plt.xlabel("Week", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_graph_dir, "incidents_by_week.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 3: Distribution by Shift ---
shift_distribution = df[shift_column].value_counts()

plt.figure(figsize=(10, 6))
shift_distribution.plot(kind="bar", color="r", edgecolor="black")
plt.title("Distribution of Incidents by Shift", fontsize=14)
plt.xlabel("Shift", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(output_graph_dir, "incidents_by_shift.png"), dpi=300, bbox_inches="tight")
plt.close()

# ----------------------------
# 6. Display graphs (optional)
# ----------------------------
print(f"Graphs saved in: {output_graph_dir}")
print("Displaying graphs...")
plt.show()  # Disable if you don't want to display the graphs

Found file: .\artifacts\ingestions\incidents\202606161613\releves_incidents_anonymised.csv
Graphs saved in: .\graphs
Displaying graphs...


6. Keep Graphs in same folder as csv input file
   - Use number of week for graph per week
   - Keep only last two digit of year

In [24]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# 1. Configure paths and environment variables
# ----------------------------
# Environment variable for the input directory
input_dir_env_var = "INPUT_DATA_DIR"
input_dir = os.getenv(input_dir_env_var, default=".")

# Name of the file to search for
file_name = "releves_incidents_anonymised.csv"
file_path = None

# Search for the file in the input directory
for root, dirs, files in os.walk(input_dir):
    if file_name in files:
        file_path = os.path.join(root, file_name)
        break

if not file_path:
    raise FileNotFoundError(f"File '{file_name}' not found in directory: {input_dir}")

print(f"Found file: {file_path}")

# Extract the directory where the CSV file is located
output_dir = os.path.dirname(file_path)

# ----------------------------
# 2. Load the CSV file
# ----------------------------
df = pd.read_csv(file_path)

# ----------------------------
# 3. Configure column names (adjust according to your CSV)
# ----------------------------
date_column = "date"          # Column containing date/time (e.g., "incident_datetime")
shift_column = "shift"        # Column containing the shift (e.g., "work_shift", "shift_type")

# Check if the columns exist
if date_column not in df.columns:
    raise ValueError(f"Column '{date_column}' not found in the CSV file. Available columns: {df.columns.tolist()}")
if shift_column not in df.columns:
    raise ValueError(f"Column '{shift_column}' not found in the CSV file. Available columns: {df.columns.tolist()}")

# Convert the date column to datetime
df[date_column] = pd.to_datetime(df[date_column], errors="coerce")

# ----------------------------
# 4. Generate the graphs and save them in the same directory as the input CSV
# ----------------------------

# --- Graph 1: Distribution by Day ---
df["day"] = df[date_column].dt.date  # Extract date (without time)
daily_distribution = df["day"].value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(daily_distribution.index, daily_distribution.values, marker="o", linestyle="-", color="b")
plt.title("Distribution of Incidents by Day", fontsize=14)
plt.xlabel("Day", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "incidents_by_day.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 2: Distribution by Week (display last 2 digits of year and week number) ---
# Extract the year and week number
df["year"] = df[date_column].dt.year
df["week_number"] = df[date_column].dt.isocalendar().week

# Create a combined label for the x-axis (e.g., "26-W24" for year 2026, week 24)
df["year_week"] = df["year"].astype(str).str[-2:] + "-W" + df["week_number"].astype(str).str.zfill(2)
weekly_distribution = df["year_week"].value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(weekly_distribution.index, weekly_distribution.values, marker="o", linestyle="-", color="g")
plt.title("Distribution of Incidents by Week", fontsize=14)
plt.xlabel("Week", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "incidents_by_week.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 3: Distribution by Shift ---
shift_distribution = df[shift_column].value_counts()

plt.figure(figsize=(10, 6))
shift_distribution.plot(kind="bar", color="r", edgecolor="black")
plt.title("Distribution of Incidents by Shift", fontsize=14)
plt.xlabel("Shift", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "incidents_by_shift.png"), dpi=300, bbox_inches="tight")
plt.close()

# ----------------------------
# 5. Display graphs (optional)
# ----------------------------
print(f"Graphs saved in: {output_dir}")
print("Displaying graphs...")
plt.show()  # Disable if you don't want to display the graphs

Found file: .\artifacts\ingestions\incidents\202606161613\releves_incidents_anonymised.csv
Graphs saved in: .\artifacts\ingestions\incidents\202606161613
Displaying graphs...


7. Histograms of incidents 
   - per signal (type of issue)
   - per machine

In [25]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# 1. Configure paths and environment variables
# ----------------------------
# Environment variable for the input directory
input_dir_env_var = "INPUT_DATA_DIR"
input_dir = os.getenv(input_dir_env_var, default=".")

# Name of the file to search for
file_name = "releves_incidents_anonymised.csv"
file_path = None

# Search for the file in the input directory
for root, dirs, files in os.walk(input_dir):
    if file_name in files:
        file_path = os.path.join(root, file_name)
        break

if not file_path:
    raise FileNotFoundError(f"File '{file_name}' not found in directory: {input_dir}")

print(f"Found file: {file_path}")

# Extract the directory where the CSV file is located
output_dir = os.path.dirname(file_path)

# ----------------------------
# 2. Load the CSV file
# ----------------------------
df = pd.read_csv(file_path)

# ----------------------------
# 3. Configure column names (adjust according to your CSV)
# ----------------------------
date_column = "date"          # Column containing date/time (e.g., "incident_datetime")
shift_column = "shift"        # Column containing the shift (e.g., "work_shift", "shift_type")
machine_column = "machine_id" # Column containing the machine ID

# Check if the required columns exist
required_columns = [date_column, shift_column, machine_column]
for column in required_columns:
    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in the CSV file. Available columns: {df.columns.tolist()}")

# Convert the date column to datetime
df[date_column] = pd.to_datetime(df[date_column], errors="coerce")

# ----------------------------
# 4. Identify all "type_*" columns (types of pannes)
# ----------------------------
type_columns = [col for col in df.columns if col.startswith("type_")]
if not type_columns:
    raise ValueError("No columns starting with 'type_' found in the CSV file.")

# ----------------------------
# 5. Filter rows where at least one "type_*" column is 1 (valid panne)
# ----------------------------
valid_pannes_mask = (df[type_columns] == 1).any(axis=1)
valid_pannes_df = df[valid_pannes_mask]

# ----------------------------
# 6. Generate the graphs and save them in the same directory as the input CSV
# ----------------------------

# --- Graph 1: Distribution by Day ---
df["day"] = df[date_column].dt.date  # Extract date (without time)
daily_distribution = df["day"].value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(daily_distribution.index, daily_distribution.values, marker="o", linestyle="-", color="b")
plt.title("Distribution of Incidents by Day", fontsize=14)
plt.xlabel("Day", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "incidents_by_day.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 2: Distribution by Week (display last 2 digits of year and week number) ---
# Extract the year and week number
df["year"] = df[date_column].dt.year
df["week_number"] = df[date_column].dt.isocalendar().week

# Create a combined label for the x-axis (e.g., "26-W24" for year 2026, week 24)
df["year_week"] = df["year"].astype(str).str[-2:] + "-W" + df["week_number"].astype(str).str.zfill(2)
weekly_distribution = df["year_week"].value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(weekly_distribution.index, weekly_distribution.values, marker="o", linestyle="-", color="g")
plt.title("Distribution of Incidents by Week", fontsize=14)
plt.xlabel("Week", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "incidents_by_week.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 3: Distribution by Shift ---
shift_distribution = df[shift_column].value_counts()

plt.figure(figsize=(10, 6))
shift_distribution.plot(kind="bar", color="r", edgecolor="black")
plt.title("Distribution of Incidents by Shift", fontsize=14)
plt.xlabel("Shift", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "incidents_by_shift.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 4: Histogram of Incidents by Machine (only valid pannes) ---
machine_distribution = valid_pannes_df[machine_column].value_counts()

plt.figure(figsize=(12, 6))
machine_distribution.plot(kind="bar", color="orange", edgecolor="black")
plt.title("Histogram of Incidents by Machine (Valid Pannes Only)", fontsize=14)
plt.xlabel("Machine ID", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "incidents_by_machine.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 5: Histogram of Incidents by Signal (only valid pannes, type_* = 1) ---
# Count incidents for each type_* column where value = 1
signal_distribution = pd.Series(dtype=int)
for type_col in type_columns:
    signal_counts = valid_pannes_df[type_col].value_counts().get(1, 0)
    signal_distribution[type_col] = signal_counts

plt.figure(figsize=(12, 6))
signal_distribution.plot(kind="bar", color="purple", edgecolor="black")
plt.title("Histogram of Incidents by Signal (Valid Pannes Only)", fontsize=14)
plt.xlabel("Signal (Type of Panne)", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "incidents_by_signal.png"), dpi=300, bbox_inches="tight")
plt.close()

# ----------------------------
# 7. Display graphs (optional)
# ----------------------------
print(f"Graphs saved in: {output_dir}")
print("Displaying graphs...")
plt.show()  # Disable if you don't want to display the graphs

Found file: .\artifacts\ingestions\incidents\202606161613\releves_incidents_anonymised.csv
Graphs saved in: .\artifacts\ingestions\incidents\202606161613
Displaying graphs...


8. Correlation graph of incidents by signal (type of issue)
   - Add library 'seaborn'
     ```
     python
     uv add seaborn
     ```

In [29]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# 1. Configure paths and environment variables
# ----------------------------
# Environment variable for the input directory
input_dir_env_var = "INPUT_DATA_DIR"
input_dir = os.getenv(input_dir_env_var, default=".")

# Name of the file to search for
file_name = "releves_incidents_anonymised.csv"
file_path = None

# Search for the file in the input directory
for root, dirs, files in os.walk(input_dir):
    if file_name in files:
        file_path = os.path.join(root, file_name)
        break

if not file_path:
    raise FileNotFoundError(f"File '{file_name}' not found in directory: {input_dir}")

print(f"Found file: {file_path}")

# Extract the directory where the CSV file is located
output_dir = os.path.dirname(file_path)

# ----------------------------
# 2. Load the CSV file
# ----------------------------
df = pd.read_csv(file_path)

# ----------------------------
# 3. Configure column names (adjust according to your CSV)
# ----------------------------
date_column = "date"          # Column containing date/time (e.g., "incident_datetime")
shift_column = "shift"        # Column containing the shift (e.g., "work_shift", "shift_type")
machine_column = "machine_id" # Column containing the machine ID
severity_column = "severity"  # Column containing the severity of incidents (e.g., "gravite")

# Check if the required columns exist
required_columns = [date_column, shift_column, machine_column, severity_column]
for column in required_columns:
    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in the CSV file. Available columns: {df.columns.tolist()}")

# Convert the date column to datetime
df[date_column] = pd.to_datetime(df[date_column], errors="coerce")

# ----------------------------
# 4. Identify all "type_*" columns (types of pannes)
# ----------------------------
type_columns = [col for col in df.columns if col.startswith("type_")]
if not type_columns:
    raise ValueError("No columns starting with 'type_' found in the CSV file.")

# ----------------------------
# 5. Filter rows where at least one "type_*" column is 1 (valid panne)
# ----------------------------
valid_pannes_mask = (df[type_columns] == 1).any(axis=1)
valid_pannes_df = df[valid_pannes_mask]

# ----------------------------
# 6. Generate the graphs and save them in the same directory as the input CSV
# ----------------------------
output_graph_dir = os.path.join(output_dir, "graphs")
os.makedirs(output_graph_dir, exist_ok=True)

# --- Graph 1: Distribution by Day ---
df["day"] = df[date_column].dt.date  # Extract date (without time)
daily_distribution = df["day"].value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(daily_distribution.index, daily_distribution.values, marker="o", linestyle="-", color="b")
plt.title("Distribution of Incidents by Day", fontsize=14)
plt.xlabel("Day", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_graph_dir, "incidents_by_day.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 2: Distribution by Week (display last 2 digits of year and week number) ---
# Extract the year and week number
df["year"] = df[date_column].dt.year
df["week_number"] = df[date_column].dt.isocalendar().week

# Create a combined label for the x-axis (e.g., "26-W24" for year 2026, week 24)
df["year_week"] = df["year"].astype(str).str[-2:] + "-W" + df["week_number"].astype(str).str.zfill(2)
weekly_distribution = df["year_week"].value_counts().sort_index()

plt.figure(figsize=(12, 6))
plt.plot(weekly_distribution.index, weekly_distribution.values, marker="o", linestyle="-", color="g")
plt.title("Distribution of Incidents by Week", fontsize=14)
plt.xlabel("Week", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_graph_dir, "incidents_by_week.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 3: Distribution by Shift ---
shift_distribution = df[shift_column].value_counts()

plt.figure(figsize=(10, 6))
shift_distribution.plot(kind="bar", color="r", edgecolor="black")
plt.title("Distribution of Incidents by Shift", fontsize=14)
plt.xlabel("Shift", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(os.path.join(output_graph_dir, "incidents_by_shift.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 4: Histogram of Incidents by Machine (only valid pannes) ---
machine_distribution = valid_pannes_df[machine_column].value_counts()

plt.figure(figsize=(12, 6))
machine_distribution.plot(kind="bar", color="orange", edgecolor="black")
plt.title("Histogram of Incidents by Machine (Valid Pannes Only)", fontsize=14)
plt.xlabel("Machine ID", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_graph_dir, "incidents_by_machine.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 5: Histogram of Incidents by Signal (only valid pannes, type_* = 1) ---
# Count incidents for each type_* column where value = 1
signal_distribution = pd.Series(dtype=int)
for type_col in type_columns:
    signal_counts = valid_pannes_df[type_col].value_counts().get(1, 0)
    signal_distribution[type_col] = signal_counts

plt.figure(figsize=(12, 6))
signal_distribution.plot(kind="bar", color="purple", edgecolor="black")
plt.title("Histogram of Incidents by Signal (Valid Pannes Only)", fontsize=14)
plt.xlabel("Signal (Type of Panne)", fontsize=12)
plt.ylabel("Number of Incidents", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_graph_dir, "incidents_by_signal.png"), dpi=300, bbox_inches="tight")
plt.close()

# --- Graph 6: Correlation Heatmap of Severity by Signal (types of pannes) ---
# Extract severity and type_* columns for valid pannes
severity_and_signals = valid_pannes_df[[severity_column] + type_columns]

# Calculate the correlation matrix
correlation_matrix = severity_and_signals.corr()

# Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap of Severity by Signal (Types of Pannes)", fontsize=14)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(output_graph_dir, "severity_correlation_by_signal.png"), dpi=300, bbox_inches="tight")
plt.close()

# ----------------------------
# 7. Display graphs (optional)
# ----------------------------
print(f"Graphs saved in: {output_graph_dir}")
print("Displaying graphs...")
plt.show()  # Disable if you don't want to display the graphs

Found file: .\artifacts\ingestions\incidents\202606161613\releves_incidents_anonymised.csv
Graphs saved in: .\artifacts\ingestions\incidents\202606161613\graphs
Displaying graphs...


In [42]:
import pandas as pd
from datetime import datetime
import os

# ----------------------------
# 0. Define environment variables 
# ----------------------------
# Environment variable for the input file path
input_file_env_var = "INPUT_FILE_PATH"
input_file_path = os.getenv(input_file_env_var, default="./artifacts/ingestions/datas/releves_incidents.csv")

# Environment variable for columns to anonymize (default: "operator_name|operator_badge")
anonymize_columns_env_var = "ANONYMIZE_COLUMNS"
anonymize_columns_str = os.getenv(anonymize_columns_env_var, default="operator_name|operator_badge")

# Environment variable for the output directory
output_dir_env_var = "OUTPUT_DIR"
output_base_dir = os.getenv(output_dir_env_var, default="./artifacts/ingestions/incidents")

# ----------------------------
# 1. Load the CSV file
# ----------------------------
# Load the CSV file
df = pd.read_csv(input_file_path)

# ----------------------------
# 2. Anonymize specified columns
# ----------------------------
# Split the string by "|" to get the list of columns
columns_to_anonymize = anonymize_columns_str.split("|")

# Replace each column with "ANONYMOUS"
for column in columns_to_anonymize:
    if column in df.columns:
        df[column] = "ANONYMOUS"
    else:
        print(f"Warning: Column '{column}' not found in the CSV file. Skipping.")

# ----------------------------
# 3. Create a directory with the date in YYYYMMDDHHMM format
# ----------------------------
# Create a subdirectory with the current date and time
date_time_output = datetime.now().strftime("%Y%m%d%H%M")  # Format YYYYMMDDHHMM
output_dir = os.path.join(output_base_dir, date_time_output)

# Create the directory if it does not exist
os.makedirs(output_dir, exist_ok=True)

# ----------------------------
# 4. Save the anonymized file in the directory
# ----------------------------
output_file_path = os.path.join(output_dir, "releves_incidents_anonymised.csv")
df.to_csv(output_file_path, index=False)

print(f"Anonymized file saved at: {output_file_path}")

Anonymized file saved at: ./artifacts/ingestions/incidents\202606171523\releves_incidents_anonymised.csv


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------
# 1. Configure paths and environment variables
# ----------------------------
# Environment variable for the input directory
input_dir_env_var = "INPUT_DATA_DIR"
input_dir = os.getenv(input_dir_env_var, default=".")

# Environment variable for the list of CSV files to process
csv_files_env_var = "CSV_FILES_TO_PROCESS"
csv_files_to_process = os.getenv(csv_files_env_var, default="").split("|")

# Check if the input directory exists
if not os.path.isdir(input_dir):
    raise FileNotFoundError(f"Directory '{input_dir}' not found.")

# ----------------------------
# 2. Define column names (adjust according to your CSV files)
# ----------------------------
date_column = "date"          # Column containing date/time
shift_column = "shift"        # Column containing the shift
machine_column = "machine_id" # Column containing the machine ID
severity_column = "severity"  # Column containing the severity of incidents

# ----------------------------
# 3. Function to process a single CSV file
# ----------------------------
def process_csv_file(file_path):
    # Load the CSV file
    df = pd.read_csv(file_path)

    # Check if the required columns exist
    required_columns = [date_column, shift_column, machine_column, severity_column]
    for column in required_columns:
        if column not in df.columns:
            raise ValueError(f"Column '{column}' not found in the CSV file: {file_path}. Available columns: {df.columns.tolist()}")

    # Convert the date column to datetime
    df[date_column] = pd.to_datetime(df[date_column], errors="coerce")

    # Identify all "type_*" columns (types of pannes)
    type_columns = [col for col in df.columns if col.startswith("type_")]
    if not type_columns:
        raise ValueError(f"No columns starting with 'type_' found in the CSV file: {file_path}")

    # Filter rows where at least one "type_*" column is 1 (valid panne)
    valid_pannes_mask = (df[type_columns] == 1).any(axis=1)
    valid_pannes_df = df[valid_pannes_mask]

    # Create output directory for graphs
    output_dir = os.path.dirname(file_path)
    output_graph_dir = os.path.join(output_dir, "graphs")
    os.makedirs(output_graph_dir, exist_ok=True)

    # Generate graphs
    generate_graphs(df, valid_pannes_df, type_columns, severity_column, machine_column, shift_column, date_column, output_graph_dir)

    print(f"Graphs generated in: {output_graph_dir}")
    print(f"Graphs generated for file: {file_path}")

# ----------------------------
# 4. Function to generate all graphs for a single CSV file
# ----------------------------
def generate_graphs(df, valid_pannes_df, type_columns, severity_column, machine_column, shift_column, date_column, output_graph_dir):
    # --- Graph 1: Distribution by Day ---
    df["day"] = df[date_column].dt.date
    daily_distribution = df["day"].value_counts().sort_index()

    plt.figure(figsize=(12, 6))
    plt.plot(daily_distribution.index, daily_distribution.values, marker="o", linestyle="-", color="b")
    plt.title("Distribution of Incidents by Day", fontsize=14)
    plt.xlabel("Day", fontsize=12)
    plt.ylabel("Number of Incidents", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "incidents_by_day.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 2: Distribution by Week ---
    df["year"] = df[date_column].dt.year
    df["week_number"] = df[date_column].dt.isocalendar().week
    df["year_week"] = df["year"].astype(str).str[-2:] + "-W" + df["week_number"].astype(str).str.zfill(2)
    weekly_distribution = df["year_week"].value_counts().sort_index()

    plt.figure(figsize=(12, 6))
    plt.plot(weekly_distribution.index, weekly_distribution.values, marker="o", linestyle="-", color="g")
    plt.title("Distribution of Incidents by Week", fontsize=14)
    plt.xlabel("Week", fontsize=12)
    plt.ylabel("Number of Incidents", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "incidents_by_week.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 3: Distribution by Shift ---
    shift_distribution = df[shift_column].value_counts()

    plt.figure(figsize=(10, 6))
    shift_distribution.plot(kind="bar", color="r", edgecolor="black")
    plt.title("Distribution of Incidents by Shift", fontsize=14)
    plt.xlabel("Shift", fontsize=12)
    plt.ylabel("Number of Incidents", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "incidents_by_shift.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 4: Histogram of Incidents by Machine (only valid pannes) ---
    machine_distribution = valid_pannes_df[machine_column].value_counts()

    plt.figure(figsize=(12, 6))
    machine_distribution.plot(kind="bar", color="orange", edgecolor="black")
    plt.title("Histogram of Incidents by Machine (Valid Pannes Only)", fontsize=14)
    plt.xlabel("Machine ID", fontsize=12)
    plt.ylabel("Number of Incidents", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "incidents_by_machine.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 5: Histogram of Incidents by Signal (only valid pannes, type_* = 1) ---
    signal_distribution = pd.Series(dtype=int)
    for type_col in type_columns:
        signal_counts = valid_pannes_df[type_col].value_counts().get(1, 0)
        signal_distribution[type_col] = signal_counts

    plt.figure(figsize=(12, 6))
    signal_distribution.plot(kind="bar", color="purple", edgecolor="black")
    plt.title("Histogram of Incidents by Signal (Valid Pannes Only)", fontsize=14)
    plt.xlabel("Signal (Type of Panne)", fontsize=12)
    plt.ylabel("Number of Incidents", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "incidents_by_signal.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 6: Correlation Heatmap of Severity by Signal (types of pannes) ---
    severity_and_signals = valid_pannes_df[[severity_column] + type_columns]
    correlation_matrix = severity_and_signals.corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title("Correlation Heatmap of Severity by Signal (Types of Pannes)", fontsize=14)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "severity_correlation_by_signal.png"), dpi=300, bbox_inches="tight")
    plt.close()

# ----------------------------
# 5. Process all CSV files specified in the environment variable or ending with "_anonymised.csv"
# ----------------------------
processed_files = set()

# If CSV_FILES_TO_PROCESS is set, use those files
if csv_files_to_process and csv_files_to_process != [""]:
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file in csv_files_to_process:
                file_path = os.path.join(root, file)
                try:
                    process_csv_file(file_path)
                    processed_files.add(file)
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

    # Check if any specified files were not found
    missing_files = set(csv_files_to_process) - processed_files
    if missing_files:
        print(f"Warning: The following files were not found: {missing_files}")
else:
    # Default behavior: process all files ending with "_anonymised.csv"
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith("_anonymised.csv"):
                file_path = os.path.join(root, file)
                try:
                    process_csv_file(file_path)
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

print("All matching CSV files processed.")

Graphs generated in : .\artifacts\ingestions\incidents\202606161613\graphs
Graphs generated for file: .\artifacts\ingestions\incidents\202606161613\releves_incidents_anonymised.csv
Graphs generated in : .\artifacts\ingestions\incidents\202606161814\graphs
Graphs generated for file: .\artifacts\ingestions\incidents\202606161814\releves_incidents_anonymised.csv
Graphs generated in : .\artifacts\ingestions\incidents\202606161817\graphs
Graphs generated for file: .\artifacts\ingestions\incidents\202606161817\releves_incidents_anonymised.csv
Graphs generated in : .\artifacts\ingestions\incidents\202606161832\graphs
Graphs generated for file: .\artifacts\ingestions\incidents\202606161832\releves_incidents_anonymised.csv
All matching CSV files processed.


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# ----------------------------
# 1. Configure paths and environment variables
# ----------------------------
# Environment variable for the input directory
input_dir_env_var = "INPUT_DATA_DIR"
input_dir = os.getenv(input_dir_env_var, default=".")

# Environment variable for the list of CSV files to process
csv_files_env_var = "CSV_FILES_TO_PROCESS"
csv_files_to_process = os.getenv(csv_files_env_var, default="").split("|")

# Check if the input directory exists
if not os.path.isdir(input_dir):
    raise FileNotFoundError(f"Directory '{input_dir}' not found.")

# ----------------------------
# 2. Define column names (adjust according to your CSV files)
# ----------------------------
# For incidents analysis
date_column = "date"          # Column containing date/time
shift_column = "shift"        # Column containing the shift
machine_column = "machine_id" # Column containing the machine ID
severity_column = "severity"  # Column containing the severity of incidents

# For telemetry data analysis
time_column = "timestamp"     # Column containing the timestamp for telemetry data
temperature_column = "temperature_c"   # Column containing temperature data
pressure_column = "pressure_bar"       # Column containing pressure data
tension_column = "voltage_mean_v"      # Column containing tension data
rotation_column = "rotation_mean_rpm"  # Column containing rotation data
production_column = "pieces_produced"  # Column containing the number of pieces produced

# ----------------------------
# 3. Function to process a single CSV file for incidents analysis
# ----------------------------
def process_incidents_csv_file(file_path):
    # Load the CSV file
    df = pd.read_csv(file_path)

    # Check if the required columns exist
    required_columns = [date_column, shift_column, machine_column, severity_column]
    for column in required_columns:
        if column not in df.columns:
            raise ValueError(f"Column '{column}' not found in the CSV file: {file_path}. Available columns: {df.columns.tolist()}")

    # Convert the date column to datetime
    df[date_column] = pd.to_datetime(df[date_column], errors="coerce")

    # Identify all "type_*" columns (types of pannes)
    type_columns = [col for col in df.columns if col.startswith("type_")]
    if not type_columns:
        raise ValueError(f"No columns starting with 'type_' found in the CSV file: {file_path}")

    # Filter rows where at least one "type_*" column is 1 (valid panne)
    valid_pannes_mask = (df[type_columns] == 1).any(axis=1)
    valid_pannes_df = df[valid_pannes_mask]

    # Create output directory for graphs
    output_dir = os.path.dirname(file_path)
    output_graph_dir = os.path.join(output_dir, "graphs")
    os.makedirs(output_graph_dir, exist_ok=True)

    # Generate graphs
    generate_incidents_graphs(df, valid_pannes_df, type_columns, severity_column, machine_column, shift_column, date_column, output_graph_dir)

    print(f"Incidents graphs generated in: {output_graph_dir}")

# ----------------------------
# 4. Function to generate all graphs for incidents analysis
# ----------------------------
def generate_incidents_graphs(df, valid_pannes_df, type_columns, severity_column, machine_column, shift_column, date_column, output_graph_dir):
    # --- Graph 1: Distribution by Day ---
    df["day"] = df[date_column].dt.date
    daily_distribution = df["day"].value_counts().sort_index()

    plt.figure(figsize=(12, 6))
    plt.plot(daily_distribution.index, daily_distribution.values, marker="o", linestyle="-", color="b")
    plt.title("Distribution of Incidents by Day", fontsize=14)
    plt.xlabel("Day", fontsize=12)
    plt.ylabel("Number of Incidents", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "incidents_by_day.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 2: Distribution by Week ---
    df["year"] = df[date_column].dt.year
    df["week_number"] = df[date_column].dt.isocalendar().week
    df["year_week"] = df["year"].astype(str).str[-2:] + "-W" + df["week_number"].astype(str).str.zfill(2)
    weekly_distribution = df["year_week"].value_counts().sort_index()

    plt.figure(figsize=(12, 6))
    plt.plot(weekly_distribution.index, weekly_distribution.values, marker="o", linestyle="-", color="g")
    plt.title("Distribution of Incidents by Week", fontsize=14)
    plt.xlabel("Week", fontsize=12)
    plt.ylabel("Number of Incidents", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "incidents_by_week.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 3: Distribution by Shift ---
    shift_distribution = df[shift_column].value_counts()

    plt.figure(figsize=(10, 6))
    shift_distribution.plot(kind="bar", color="r", edgecolor="black")
    plt.title("Distribution of Incidents by Shift", fontsize=14)
    plt.xlabel("Shift", fontsize=12)
    plt.ylabel("Number of Incidents", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "incidents_by_shift.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 4: Histogram of Incidents by Machine (only valid pannes) ---
    machine_distribution = valid_pannes_df[machine_column].value_counts()

    plt.figure(figsize=(12, 6))
    machine_distribution.plot(kind="bar", color="orange", edgecolor="black")
    plt.title("Histogram of Incidents by Machine (Valid Pannes Only)", fontsize=14)
    plt.xlabel("Machine ID", fontsize=12)
    plt.ylabel("Number of Incidents", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "incidents_by_machine.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 5: Histogram of Incidents by Signal (only valid pannes, type_* = 1) ---
    signal_distribution = pd.Series(dtype=int)
    for type_col in type_columns:
        signal_counts = valid_pannes_df[type_col].value_counts().get(1, 0)
        signal_distribution[type_col] = signal_counts

    plt.figure(figsize=(12, 6))
    signal_distribution.plot(kind="bar", color="purple", edgecolor="black")
    plt.title("Histogram of Incidents by Signal (Valid Pannes Only)", fontsize=14)
    plt.xlabel("Signal (Type of Panne)", fontsize=12)
    plt.ylabel("Number of Incidents", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "incidents_by_signal.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 6: Correlation Heatmap of Severity by Signal (types of pannes) ---
    severity_and_signals = valid_pannes_df[[severity_column] + type_columns]
    correlation_matrix = severity_and_signals.corr()

    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
    plt.title("Correlation Heatmap of Severity by Signal (Types of Pannes)", fontsize=14)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "severity_correlation_by_signal.png"), dpi=300, bbox_inches="tight")
    plt.close()

# ----------------------------
# 5. Function to process the telemetry CSV file
# ----------------------------
def process_telemetry_csv_file(file_path):
    # Load the CSV file
    df = pd.read_csv(file_path)

    # Create output directory for graphs
    # output_dir = os.path.dirname(file_path)
    output_dir = os.path.dirname("./artifacts/ingestions/")
    output_graph_dir = os.path.join(output_dir, "telemetry", datetime.now().strftime("%Y%m%d%H%M"), "graphs")
    os.makedirs(output_graph_dir, exist_ok=True)

    # Generate graphs
    generate_telemetry_graphs(df, time_column, temperature_column, pressure_column, tension_column, rotation_column, production_column, output_graph_dir)

    print(f"Telemetry graphs generated in: {output_graph_dir}")

# ----------------------------
# 6. Function to generate all graphs for telemetry data
# ----------------------------
def generate_telemetry_graphs(df, time_column, temperature_column, pressure_column, tension_column, rotation_column, production_column, output_graph_dir):
    # Check if the required columns exist
    required_columns = [time_column, temperature_column, pressure_column, tension_column, rotation_column, production_column]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        print(f"Warning: The following columns are missing in the CSV file: {missing_columns}. Skipping telemetry graphs.")
        return

    # Convert the time column to datetime
    df[time_column] = pd.to_datetime(df[time_column], errors="coerce")

    # --- Graph 1: Temperature Distribution Over Time ---
    plt.figure(figsize=(12, 6))
    plt.plot(df[time_column], df[temperature_column], marker="o", linestyle="-", color="red", label="Temperature")
    plt.title("Temperature Distribution Over Time", fontsize=14)
    plt.xlabel("Time", fontsize=12)
    plt.ylabel("Temperature", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "temperature_over_time.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 2: Pressure Distribution Over Time ---
    plt.figure(figsize=(12, 6))
    plt.plot(df[time_column], df[pressure_column], marker="o", linestyle="-", color="blue", label="Pressure")
    plt.title("Pressure Distribution Over Time", fontsize=14)
    plt.xlabel("Time", fontsize=12)
    plt.ylabel("Pressure", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "pressure_over_time.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 3: Tension Distribution Over Time ---
    plt.figure(figsize=(12, 6))
    plt.plot(df[time_column], df[tension_column], marker="o", linestyle="-", color="green", label="Tension")
    plt.title("Tension Distribution Over Time", fontsize=14)
    plt.xlabel("Time", fontsize=12)
    plt.ylabel("Tension", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "tension_over_time.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 4: Rotation Distribution Over Time ---
    plt.figure(figsize=(12, 6))
    plt.plot(df[time_column], df[rotation_column], marker="o", linestyle="-", color="purple", label="Rotation")
    plt.title("Rotation Distribution Over Time", fontsize=14)
    plt.xlabel("Time", fontsize=12)
    plt.ylabel("Rotation", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "rotation_over_time.png"), dpi=300, bbox_inches="tight")
    plt.close()

    # --- Graph 5: Pieces Produced Over Time ---
    plt.figure(figsize=(12, 6))
    plt.plot(df[time_column], df[production_column], marker="o", linestyle="-", color="orange", label="Pieces Produced")
    plt.title("Pieces Produced Over Time", fontsize=14)
    plt.xlabel("Time", fontsize=12)
    plt.ylabel("Pieces Produced", fontsize=12)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_graph_dir, "pieces_produced_over_time.png"), dpi=300, bbox_inches="tight")
    plt.close()

# ----------------------------
# 7. Process all CSV files in the input directory
# ----------------------------
processed_files = set()

# If CSV_FILES_TO_PROCESS is set, use those files
if csv_files_to_process and csv_files_to_process != [""]:
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file in csv_files_to_process and file.endswith(".csv"):
                file_path = os.path.join(root, file)
                try:
                    if file == "telemetry.csv":
                        process_telemetry_csv_file(file_path)
                    elif file.endswith("_anonymised.csv"):
                        process_incidents_csv_file(file_path)
                    processed_files.add(file)
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

    # Check if any specified files were not found
    missing_files = set(csv_files_to_process) - processed_files
    if missing_files:
        print(f"Warning: The following files were not found: {missing_files}")
else:
    # Default behavior: process all files ending with "_anonymised.csv" or named "telemetry.csv"
    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".csv"):
                file_path = os.path.join(root, file)
                try:
                    if file == "telemetry.csv":
                        process_telemetry_csv_file(file_path)
                    elif file.endswith("_anonymised.csv"):
                        process_incidents_csv_file(file_path)
                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

print("All matching CSV files processed.")